In [1]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

In [ ]:
#    import dotenv
#
#    project_dir = os.path.join(os.path.dirname(__file__), os.pardir)
#    dotenv_path = os.path.join(project_dir, '.env')
#    dotenv.load_dotenv(dotenv_path)
#   

In [24]:
from dotenv import load_dotenv
import os

load_dotenv("../.env")

True

In [4]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
593,"Now I love Bela Lugosi,don't get me wrong,he i...",negative
345,Routine suspense yarn about a sociopath (Dillo...,negative
946,1st watched 5/17/2002 - 3 out of 10(Dir-Ewald ...,negative
519,"""Most of us at least inhabit two worlds , the ...",positive
622,Excellent performance. There still are good ac...,positive


In [5]:
# data preprocessing
from nltk.corpus import stopwords

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

<>:33: SyntaxWarning: invalid escape sequence '\s'
<>:33: SyntaxWarning: invalid escape sequence '\s'
C:\Users\Dell\AppData\Local\Temp\ipykernel_4500\2950656902.py:33: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub('\s+', ' ', text).strip()


In [7]:
df = normalize_text(df)
df.head()

,review,sentiment
593,love bela lugosi get wrong one interesting peo...,negative
345,routine suspense yarn sociopath dillon give sp...,negative
946,st watched dir ewald andre dupont fairly lame ...,negative
519,u least inhabit two world real world mercy cir...,positive
622,excellent performance still good actor around ...,positive


In [8]:
df['sentiment'].value_counts()

sentiment
negative    262
positive    238
Name: count, dtype: int64

In [9]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [10]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
593,love bela lugosi get wrong one interesting peo...,0
345,routine suspense yarn sociopath dillon give sp...,0
946,st watched dir ewald andre dupont fairly lame ...,0
519,u least inhabit two world real world mercy cir...,1
622,excellent performance still good actor around ...,1


In [11]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [13]:
vectorizer = CountVectorizer(max_features=100) #BoW 
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [36]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
X_train.shape, y_train.shape

((375, 100), (375,))

## Mlfow dagshub setup

In [29]:
from dotenv import load_dotenv
import os

load_dotenv("../.env") # load env

True

In [51]:
import dagshub
import mlflow

mlflow.set_tracking_uri(os.getenv("MLFLOW_DAGSHUB_TRACKING_URI"))

dagshub.init(repo_owner= os.getenv("REPO_OWNER"), 
             repo_name=os.getenv("REPO_NAME"),
             mlflow=True)

mlflow.set_experiment("LogisticRegression Base Model")

2025-07-07 17:03:39,296 - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/shubhamair1996/mlops-pipeline "HTTP/1.1 200 OK"


Initialized MLflow to track repo "shubhamair1996/mlops-pipeline"

2025-07-07 17:03:39,305 - INFO - Initialized MLflow to track repo "shubhamair1996/mlops-pipeline"


Repository shubhamair1996/mlops-pipeline initialized!

2025-07-07 17:03:39,308 - INFO - Repository shubhamair1996/mlops-pipeline initialized!
2025/07/07 17:03:39 INFO mlflow.tracking.fluent: Experiment with name 'LogisticRegression Base Model' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/2862d6969b274fc1b412af4b251f5b56', creation_time=1751888019982, experiment_id='1', last_update_time=1751888019982, lifecycle_stage='active', name='LogisticRegression Base Model', tags={}>

In [47]:
X_train[:1].toarray(), X_train[:1].toarray().shape, y_train[0]# this are the sparse metrics 

(array([[ 1,  0,  1,  0,  2,  0,  0,  1,  0,  0,  0,  2,  1, 10,  1,  3,
          1,  0,  0,  0,  4,  1,  0,  0,  3,  0,  4,  1,  0,  0,  1,  1,
          1,  1,  0,  1,  0,  0,  0,  2,  0,  1,  0,  0,  2,  0,  0,  0,
          1,  2,  2,  2,  1,  0,  3,  0,  2,  0,  0,  0,  0,  1,  0,  2,
          0,  1,  2,  0,  2,  0,  1,  0,  2,  1,  0,  4,  0,  1,  0,  0,
          0,  2,  0,  1,  0,  3,  2,  2,  1,  0,  1,  0,  1,  0,  2,  0,
          0,  0,  2,  0]]),
 (1, 100),
 np.int64(0))

In [ ]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test_size", 0.25)
        mlflow.log_param("maximum_iteration", 1500)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        # mlflow.log_metrics({
        #     "accuracy": accuracy,
        #     "precision": precision,
        #     "recall": recall,
        #     "f1_score": f1
        # })

        logging.info("Saving and logging the model...")
        # mlflow.sklearn.log_model(model, "model")
        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path="model"
        )
        
        # mlflow.log_artifact("path/to/file")
        #  mlflow.log_artifact("data.csv", artifact_path="dataset")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2025-07-07 17:13:06,785 - INFO - Starting MLflow run...


2025-07-07 17:13:10,896 - INFO - Logging preprocessing parameters...
2025-07-07 17:13:12,849 - INFO - Initializing Logistic Regression model...
2025-07-07 17:13:12,850 - INFO - Fitting the model...
2025/07/07 17:13:12 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2025/07/07 17:13:15 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during sklearn autologging: INTERNAL_ERROR: Response: {'error': 'unsupported endpoint, please contact support@dagshub.com'}
2025-07-07 17:13:15,047 - INFO - Model training complete.
2025-07-07 17:13:15,048 - INFO - Logging model parameters...
2025-07-07 17:13:15,415 - INFO - Making predictions...
2025-07-07 17:13:15,417 - INFO - Calculating evaluation metrics...
2025-07-07 17:13:15,430 - INFO - Logging evaluation metrics...
2025-07-07 17:13:20,263 - INFO - Saving and logging the model...
2025/07/07 17:13:20 WARNING mlflow.models.model: `artifact_pat

🏃 View run serious-doe-10 at: https://dagshub.com/shubhamair1996/mlops-pipeline.mlflow/#/experiments/1/runs/a501dd74871545b4bd4e5db201275b38
🧪 View experiment at: https://dagshub.com/shubhamair1996/mlops-pipeline.mlflow/#/experiments/1


| What You Did                               | What’s the Problem                                     | Fix                                     |
| ------------------------------------------ | ------------------------------------------------------ | --------------------------------------- |
| Used `log_model()` with DagsHub MLflow URI | DagsHub doesn’t support artifact upload via MLflow yet | Use local tracking or DVC for artifacts |
